## 1. Import required Libraries

In [1]:
!pip install xgboost scikit-learn pandas cleanlab
!pip install importnb
!pip install tensorflow
!pip install opencv-contrib-python
!pip install pillow-heif

In [1]:
import tensorflow as tf
import cv2
import cleanlab
import shutil
import os
from PIL import Image
import pillow_heif
import numpy as np
from collections import Counter
import pandas as pd

## 2. Loading the Data

In [3]:
# ==============================================
# 1. PATH SETUP
# ==============================================
repo_path = 'C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository'
dataset_path = 'C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/dataset'

In [4]:
processed_image_count = 0
count_label_loading_error = 0

os.makedirs(dataset_path, exist_ok=True)

output_file = 'image_labels.txt'

labels = []
corrupted_images = []
unlabeled_images = []

print("===== COPYING IMAGES & ASSIGNING LABELS =====\n")

# ==============================================
# 2. MAIN LOOP: COPY + LABEL IN ONE GO
# ==============================================

label_map = {}  # StageN → N

for folder_name in sorted(os.listdir(repo_path)):
    source_folder = os.path.join(repo_path, folder_name)

    if not os.path.isdir(source_folder):
        continue  # skip files in root

    # ----------------------------------------
    # Detect folders in StageN format (stage1, stage2, ...)
    # ----------------------------------------
    if folder_name.lower().startswith("stage"):
        try:
            stage_number = int(folder_name[5:])
            label_map[folder_name] = stage_number
        except:
            print(f"⚠️ Warning: Folder {folder_name} is not a valid StageN.")
            continue

        current_label_int = label_map[folder_name]

        # ----------------------------------------
        # Iterate images and process them immediately
        # ----------------------------------------
        for filename in os.listdir(source_folder):

            source_file = os.path.join(source_folder, filename)

            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                # Do not use source_file before declaring!
                print(f"⚠️ Skipping non-image file: {source_file}")
                count_label_loading_error += 1
                continue

            img = cv2.imread(source_file)

            if img is None:
                corrupted_images.append(os.path.join(folder_name, filename))
                print(f"❌ Cannot read (corrupted): {source_file}")
                count_label_loading_error += 1
                continue

            # Copy valid image to dataset
            dst = os.path.join(dataset_path, filename)
            shutil.copy2(source_file, dst)
            processed_image_count += 1

            # Assign label immediately
            labels.append((filename, current_label_int-1))

    # ----------------------------------------
    # Folder not StageN → images not labeled
    # ----------------------------------------
    else:
        for filename in os.listdir(source_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                unlabeled_images.append(os.path.join(folder_name, filename))


print("\nTotal successfully processed images: ", processed_image_count)
print("Number of unreadable images: ", count_label_loading_error)
print("Total images scanned: ", processed_image_count + count_label_loading_error)
print("\n===== FINISHED COPYING & LABELING =====\n")

# ==============================================
# 3. SAVE LABEL FILE
# ==============================================

with open(output_file, "w", encoding="utf-8") as f:
    for filename, label in labels:
        f.write(f"{filename}: {label}\n")

print(f"🎯 Saved {len(labels)} labeled images to {output_file}\n")


# ==============================================
# 4. REPORT
# ==============================================

if corrupted_images:
    print("❌ Corrupted images:")
    for x in corrupted_images:
        print(" -", x)

if unlabeled_images:
    print("\n⚠️ Images in non-Stage folders (not labeled):")
    for x in unlabeled_images:
        print(" -", x)

if not corrupted_images and not unlabeled_images:
    print("🎉 All images were valid and assigned labels correctly!")

===== COPYING IMAGES & ASSIGNING LABELS =====

⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1 _ 7 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_ 10 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_1 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_2 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of Huddersfield/Data-driven-AI/Assignment 2/CHS2406_Coursework2_Data_Repository\Stage1\stage 1_3 u2362759.heic
⚠️ Skipping non-image file: C:/Users/Admin/OneDrive - University of H

In [5]:
os.makedirs(dataset_path, exist_ok=True)

output_file = 'image_labels.txt'

labels = []
corrupted_images = []
unlabeled_images = []

processed_image_count = 0
converted_heic_count = 0

# NEW: Count HEIC images per stage
heic_count_per_stage = {}

print("===== CONVERT HEIC ONLY + COPYING TO DATASET =====\n")

# ===============================
# 2. FUNCTION: CONVERT HEIC
# ===============================
def convert_heic_to_jpg(source_file):
    """Convert HEIC to JPG, return: OpenCV image + new filename"""
    try:
        heif_file = pillow_heif.read_heif(source_file)
        img = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data)
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        filename_new = os.path.splitext(os.path.basename(source_file))[0] + ".jpg"
        return img_cv, filename_new
    except:
        return None, None

# ===============================
# 3. MAIN LOOP: PROCESS HEIC ONLY
# ===============================
for folder_name in sorted(os.listdir(repo_path)):
    source_folder = os.path.join(repo_path, folder_name)
    if not os.path.isdir(source_folder):
        continue

    # Process StageN folders
    if folder_name.lower().startswith("stage"):
        try:
            stage_number = int(folder_name[5:])
        except:
            print(f"⚠️ Invalid folder (not StageN): {folder_name}")
            continue

        current_label_int = stage_number  # stage N → label = N-1

        # NEW: Initialize counter for this stage
        if stage_number not in heic_count_per_stage:
            heic_count_per_stage[stage_number] = 0

        for filename in os.listdir(source_folder):
            source_file = os.path.join(source_folder, filename)
            ext = filename.lower().split('.')[-1]

            if ext != 'heic':
                continue  # Skip non-HEIC images

            # Convert HEIC
            img, new_filename = convert_heic_to_jpg(source_file)
            if img is None:
                corrupted_images.append(os.path.join(folder_name, filename))
                continue

            # Count HEIC images
            converted_heic_count += 1
            heic_count_per_stage[stage_number] += 1

            # Save to dataset
            dst = os.path.join(dataset_path, new_filename)
            cv2.imwrite(dst, img)

            processed_image_count += 1

            # Assign stage label → label = stage - 1
            labels.append((new_filename, current_label_int - 1))

    else:
        # HEIC files outside StageN → not labeled
        for filename in os.listdir(source_folder):
            ext = filename.lower().split('.')[-1]
            if ext == 'heic':
                unlabeled_images.append(os.path.join(folder_name, filename))

# ===============================
# 4. SAVE LABEL FILE
# ===============================
with open(output_file, "a", encoding="utf-8") as f:
    for filename, label in labels:
        f.write(f"{filename}: {label}\n")

# ===============================
# 5. REPORT
# ===============================
print("\n================= HEIC COUNT BY STAGE =================\n")
for stage, count in sorted(heic_count_per_stage.items()):
    print(f"Stage {stage}: {count} HEIC converted")

print("\n========================================================")
print(f"🎯 Total HEIC converted and copied: {converted_heic_count}")
print(f"📌 Total HEIC successfully processed: {processed_image_count}")

if corrupted_images:
    print("\n❌ Corrupted HEIC files:")
    for x in corrupted_images:
        print(" -", x)

if unlabeled_images:
    print("\n⚠️ HEIC files in non-Stage folders (NOT labeled):")
    for x in unlabeled_images:
        print(" -", x)

print(f"\n📄 Saved labels to: {output_file}")
print("\n✅ Done.")

===== CONVERT HEIC ONLY + COPYING TO DATASET =====


================= HEIC COUNT BY STAGE =================

Stage 1: 20 HEIC converted
Stage 2: 20 HEIC converted
Stage 3: 20 HEIC converted
Stage 4: 30 HEIC converted
Stage 5: 20 HEIC converted
Stage 6: 19 HEIC converted
Stage 7: 20 HEIC converted
Stage 8: 20 HEIC converted

🎯 Total HEIC converted and copied: 169
📌 Total HEIC successfully processed: 169

📄 Saved labels to: image_labels.txt

✅ Done.


## 3. Data Labelling Errors

## Label errors:
**Explain what kind of errors you found in the dataset.**
1. Image not related to the topics
2. Wrong pose
3. the shooting angle which haven't many image same
4. image from AI
5. image which have a lot of items (noisy)
6. much bubble cover the hand
7. include the body
8. tay mo, nhieu
9. plane white image
10. The hand is too small
11. the hand is not in the center, just a part of hand appear in the image

**List the total number of images left in each class/stage after the label error handling**

<br>

<ol>
  <li>Stage 1: <<Number of images>></li>
  <li>Stage 2: <<Number of images>></li>
  <li>Stage 3: <<Number of images>></li>
  <li>Stage 4: <<Number of images>></li>
  <li>Stage 5: <<Number of images>></li>
  <li>Stage 6: <<Number of images>></li>
  <li>Stage 7: <<Number of images>></li>
  <li>Stage 8: <<Number of images>></li>
</ol>

## 4. Pre-process the Dataset

In [ ]:
# ===============================# Part 4: Pre-process the Dataset# ===============================import tensorflow as tffrom tensorflow.keras.applications.mobilenet_v2 import preprocess_input# Configurationlabels_file = "image_labels.txt"dataset_path = "dataset"  # folder containing copied imagesrequired_size = (150, 150)print("=" * 80)print("PART 4: PRE-PROCESSING THE DATASET")print("=" * 80)# ===============================# 1) Load labels from file# ===============================print("\n📂 Loading labels from file...")labels_data = []if not os.path.exists(labels_file):    raise FileNotFoundError(f"File {labels_file} not found. Please run the cell to create labels first.")with open(labels_file, "r", encoding="utf-8") as f:    for line in f:        parts = line.strip().split(": ")        if len(parts) == 2:            fname, lbl = parts[0], int(parts[1])            labels_data.append((fname, lbl))print(f"✅ Loaded {len(labels_data)} labels from {labels_file}")# ===============================# 2) Load and preprocess images# ===============================print(f"\n🖼️  Loading and preprocessing images to size {required_size}...")X_images = []y_labels = []skipped_count = 0loaded_count = 0for filename, label in labels_data:    img_path = os.path.join(dataset_path, filename)        if not os.path.exists(img_path):        skipped_count += 1        continue        try:        # Load image using Pillow        with Image.open(img_path) as img:            # Convert to RGB (in case of RGBA or grayscale)            img_rgb = img.convert('RGB')                        # Resize to required size            img_resized = img_rgb.resize(required_size, Image.LANCZOS)                        # Convert to numpy array            img_array = np.array(img_resized, dtype=np.float32)                        X_images.append(img_array)            y_labels.append(label)            loaded_count += 1                        if loaded_count % 1000 == 0:                print(f"  Loaded {loaded_count} images...")                    except Exception as e:        print(f"⚠️  Error loading {filename}: {e}")        skipped_count += 1        continue# Convert to numpy arraysX_images = np.array(X_images, dtype=np.float32)y_labels = np.array(y_labels, dtype=np.int32)print(f"\n✅ Successfully loaded {loaded_count} images")print(f"⚠️  Skipped {skipped_count} images (not found or corrupted)")# ===============================# 3) Normalize using MobileNetV2 preprocessing# ===============================print("\n🔧 Applying MobileNetV2 preprocessing (normalization)...")print(f"   Input shape before preprocessing: {X_images.shape}")print(f"   Pixel value range: [{X_images.min():.2f}, {X_images.max():.2f}]")# Apply MobileNetV2 preprocessing (scales to [-1, 1])X_preprocessed = preprocess_input(X_images)print(f"   Preprocessed shape: {X_preprocessed.shape}")print(f"   Preprocessed pixel range: [{X_preprocessed.min():.2f}, {X_preprocessed.max():.2f}]")# ===============================# 4) Display class distribution# ===============================print("\n📊 Class Distribution:")print("-" * 40)unique, counts = np.unique(y_labels, return_counts=True)for stage, count in zip(unique, counts):    print(f"  Stage {stage + 1} (Label {stage}): {count} images")print("\n" + "=" * 80)print("✨ PART 4 COMPLETED: Dataset preprocessed successfully!")print("=" * 80)

## 5. Split the data
<br>

Split the data into training, validation and testing dataset using Stratification, ensuring equal class distribution.

Choose appropriate values of training, validation and testing datasets.

Display total number of images in each dataset split.

In [ ]:
# ===============================# Part 5: Split the Data# ===============================from sklearn.model_selection import train_test_splitprint("=" * 80)print("PART 5: SPLITTING THE DATASET")print("=" * 80)# ===============================# Stratified Split: 70% Train, 15% Val, 15% Test# ===============================print("\n📊 Performing stratified split...")print("   Training: 70%, Validation: 15%, Testing: 15%")# First split: 70% train, 30% temp (which will be split into val and test)X_train, X_temp, y_train, y_temp = train_test_split(    X_preprocessed, y_labels,    test_size=0.30,    stratify=y_labels,    random_state=42)# Second split: Split temp into 50-50 (15% and 15% of original)X_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp,    test_size=0.50,    stratify=y_temp,    random_state=42)# ===============================# Display split information# ===============================print("\n✅ Split completed successfully!")print("\n📈 Dataset Split Summary:")print("-" * 40)print(f"  Training set:   {len(X_train):5d} images ({len(X_train)/len(X_preprocessed)*100:.1f}%)")print(f"  Validation set: {len(X_val):5d} images ({len(X_val)/len(X_preprocessed)*100:.1f}%)")print(f"  Testing set:    {len(X_test):5d} images ({len(X_test)/len(X_preprocessed)*100:.1f}%)")print(f"  Total:          {len(X_preprocessed):5d} images")# Display class distribution in each splitprint("\n📊 Class Distribution per Split:")print("-" * 40)for split_name, split_labels in [("Training", y_train), ("Validation", y_val), ("Testing", y_test)]:    print(f"\n{split_name} Set:")    unique, counts = np.unique(split_labels, return_counts=True)    for stage, count in zip(unique, counts):        print(f"  Stage {stage + 1} (Label {stage}): {count} images")print("\n" + "=" * 80)print("✨ PART 5 COMPLETED: Data split successfully with stratification!")print("=" * 80)

## 6. Model Implementation

In [ ]:
# ===============================# Part 6: Model Implementation# ===============================from tensorflow.keras.applications import MobileNetV2from xgboost import XGBClassifierprint("=" * 80)print("PART 6: MODEL IMPLEMENTATION")print("=" * 80)# ===============================# 6.1) Feature Extraction using MobileNetV2# ===============================print("\n🧠 Loading MobileNetV2 for feature extraction...")print("   Weights: ImageNet")print("   Configuration: include_top=False, pooling='avg'")print("   Input shape: (150, 150, 3)")# Load pre-trained MobileNetV2base_model = MobileNetV2(    weights="imagenet",    include_top=False,    pooling="avg",    input_shape=(150, 150, 3))print("\n✅ MobileNetV2 loaded successfully!")print(f"   Output feature dimension: {base_model.output_shape[1]}")# Extract features from all splitsprint("\n🔄 Extracting features from training set...")X_train_features = base_model.predict(X_train, batch_size=32, verbose=1)print("\n🔄 Extracting features from validation set...")X_val_features = base_model.predict(X_val, batch_size=32, verbose=1)print("\n🔄 Extracting features from test set...")X_test_features = base_model.predict(X_test, batch_size=32, verbose=1)print(f"\n✅ Feature extraction completed!")print(f"   Training features shape:   {X_train_features.shape}")print(f"   Validation features shape: {X_val_features.shape}")print(f"   Testing features shape:    {X_test_features.shape}")# ===============================# 6.2) Train XGBoost Classifier# ===============================print("\n" + "=" * 80)print("🚀 Training XGBoost Classifier...")print("=" * 80)# Initialize XGBoost Classifier with optimized hyperparametersxgb_clf = XGBClassifier(    n_estimators=1000,    learning_rate=0.05,    max_depth=6,    subsample=0.8,    colsample_bytree=0.8,    objective="multi:softprob",    eval_metric="mlogloss",    early_stopping_rounds=10,    random_state=42,    n_jobs=-1,  # Use all CPU cores    verbosity=1)print("\nHyperparameters:")print(f"  n_estimators: 1000")print(f"  learning_rate: 0.05")print(f"  max_depth: 6")print(f"  subsample: 0.8")print(f"  colsample_bytree: 0.8")print(f"  early_stopping_rounds: 10")# Train with validation set for early stoppingprint("\n⏳ Training in progress (this may take a few minutes)...")xgb_clf.fit(    X_train_features, y_train,    eval_set=[(X_val_features, y_val)],    verbose=50  # Print every 50 iterations)print("\n✅ Training completed!")print(f"   Best iteration: {xgb_clf.best_iteration}")print(f"   Best score (mlogloss): {xgb_clf.best_score:.4f}")print("\n" + "=" * 80)print("✨ PART 6 COMPLETED: Model trained successfully!")print("=" * 80)

## 7. Evaluate the Model

In [ ]:
# ===============================# Part 7: Evaluate the Model# ===============================from sklearn.metrics import accuracy_score, classification_report, confusion_matriximport seaborn as snsimport matplotlib.pyplot as pltprint("=" * 80)print("PART 7: MODEL EVALUATION")print("=" * 80)# ===============================# 7.1) Make Predictions# ===============================print("\n🔮 Making predictions on test set...")y_pred = xgb_clf.predict(X_test_features)# ===============================# 7.2) Calculate Metrics# ===============================print("\n" + "=" * 80)print("📊 EVALUATION METRICS")print("=" * 80)# Accuracytest_accuracy = accuracy_score(y_test, y_pred)print(f"\n🎯 Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")# Classification Reportprint("\n📋 Classification Report:")print("=" * 80)class_names = [f"Stage {i+1}" for i in range(8)]print(classification_report(y_test, y_pred, target_names=class_names, digits=4))# ===============================# 7.3) Confusion Matrix# ===============================print("\n" + "=" * 80)print("📊 Confusion Matrix")print("=" * 80)cm = confusion_matrix(y_test, y_pred)# Plot confusion matrixplt.figure(figsize=(10, 8))sns.heatmap(    cm,    annot=True,    fmt='d',    cmap='Blues',    xticklabels=class_names,    yticklabels=class_names,    cbar_kws={'label': 'Count'})plt.title('Confusion Matrix - Hand Washing Stage Classification', fontsize=14, fontweight='bold')plt.xlabel('Predicted Stage', fontsize=12)plt.ylabel('True Stage', fontsize=12)plt.tight_layout()plt.show()print("\n✅ Confusion matrix plotted successfully!")

In [ ]:
# ===============================# 7.4) Training Curves# ===============================print("\n" + "=" * 80)print("📈 Training Curves")print("=" * 80)# Get training historyresults = xgb_clf.evals_result()train_logloss = results['validation_0']['mlogloss']# Plot training curveplt.figure(figsize=(10, 6))plt.plot(range(len(train_logloss)), train_logloss, 'b-', linewidth=2, label='Validation Log Loss')plt.axvline(x=xgb_clf.best_iteration, color='r', linestyle='--', linewidth=2, label=f'Best Iteration ({xgb_clf.best_iteration})')plt.xlabel('Iteration', fontsize=12)plt.ylabel('Log Loss', fontsize=12)plt.title('XGBoost Training History - Log Loss over Iterations', fontsize=14, fontweight='bold')plt.legend(fontsize=10)plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"\n✅ Training converged at iteration {xgb_clf.best_iteration}")print(f"   Best validation log loss: {min(train_logloss):.4f}")

In [ ]:
# ===============================# 7.5) Make Inference on Random Test Images# ===============================print("\n" + "=" * 80)print("🔍 Inference on Random Test Images")print("=" * 80)# Select random test imagesnp.random.seed(42)num_samples = 8random_indices = np.random.choice(len(X_test), num_samples, replace=False)# Make predictionssample_features = X_test_features[random_indices]sample_predictions = xgb_clf.predict(sample_features)sample_true_labels = y_test[random_indices]# Denormalize images for display (reverse MobileNetV2 preprocessing)# MobileNetV2 preprocessing scales to [-1, 1], so we reverse itsample_images = X_test[random_indices].copy()# Reverse: x = (x / 127.5) - 1.0  =>  original = (x + 1.0) * 127.5sample_images_display = (sample_images + 1.0) * 127.5sample_images_display = np.clip(sample_images_display, 0, 255).astype(np.uint8)# Plot resultsfig, axes = plt.subplots(2, 4, figsize=(16, 8))axes = axes.ravel()for idx in range(num_samples):    ax = axes[idx]        img = sample_images_display[idx]    true_label = sample_true_labels[idx]    pred_label = sample_predictions[idx]        ax.imshow(img)    ax.axis('off')        # Color code: green if correct, red if incorrect    color = 'green' if true_label == pred_label else 'red'    title = f"True: Stage {true_label + 1}\nPred: Stage {pred_label + 1}"    ax.set_title(title, fontsize=10, fontweight='bold', color=color)plt.suptitle('Random Test Image Predictions', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()# Calculate accuracy for these samplescorrect = np.sum(sample_true_labels == sample_predictions)print(f"\n✅ Prediction accuracy on {num_samples} random samples: {correct}/{num_samples} ({correct/num_samples*100:.1f}%)")print("\n" + "=" * 80)print("✨ PART 7 COMPLETED: Model evaluation finished successfully!")print("=" * 80)